In [ ]:
!pip install statsmodels

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# PULL DATA
# ============================================================

TICKER_1 = "EURGBP=X"
TICKER_2 = "EURUSD=X"
PERIOD   = "2y"

data1 = yf.download(TICKER_1, period=PERIOD, auto_adjust=True)["Close"].squeeze()
data2 = yf.download(TICKER_2, period=PERIOD, auto_adjust=True)["Close"].squeeze()

# Align dates
prices = pd.DataFrame({"EURGBP": data1, "EURUSD": data2}).dropna()

print(f"Data loaded: {len(prices)} trading days")
print(f"\n  EURGBP — Start: {prices['EURGBP'].iloc[0]:.4f}  "
      f"End: {prices['EURGBP'].iloc[-1]:.4f}  "
      f"Return: {(prices['EURGBP'].iloc[-1]/prices['EURGBP'].iloc[0]-1):.2%}")
print(f"  EURUSD — Start: {prices['EURUSD'].iloc[0]:.4f}  "
      f"End: {prices['EURUSD'].iloc[-1]:.4f}  "
      f"Return: {(prices['EURUSD'].iloc[-1]/prices['EURUSD'].iloc[0]-1):.2%}")

# ============================================================
# NORMALISE TO 100 FOR COMPARISON
# ============================================================

norm = prices / prices.iloc[0] * 100

# ============================================================
# PLOT
# ============================================================

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Normalised Price (rebased to 100)", "Price Spread (EURGBP - EURUSD)"),
    vertical_spacing=0.12,
    row_heights=[0.6, 0.4]
)

fig.add_trace(go.Scatter(
    x=norm.index, y=norm["EURGBP"],
    mode="lines", name="EURGBP",
    line=dict(color="#4a9edd", width=1.5),
    hovertemplate="%{x}<br>EURGBP: %{y:.2f}<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=norm.index, y=norm["EURUSD"],
    mode="lines", name="EURUSD",
    line=dict(color="#f39c12", width=1.5),
    hovertemplate="%{x}<br>EURUSD: %{y:.2f}<extra></extra>"
), row=1, col=1)

spread = prices["EURGBP"] - prices["EURUSD"]
spread_mean = spread.mean()

fig.add_trace(go.Scatter(
    x=spread.index, y=spread,
    mode="lines", name="Spread",
    line=dict(color="#28a745", width=1.2),
    hovertemplate="%{x}<br>Spread: %{y:.4f}<extra></extra>"
), row=2, col=1)

fig.add_hline(y=spread_mean, line_dash="dash",
              line_color="white", line_width=1,
              annotation_text=f"Mean: {spread_mean:.4f}",
              annotation_font_color="white", row=2, col=1)

fig.update_layout(
    title=dict(text="EURGBP vs EURUSD — Pairs Trading Analysis", x=0.5,
               font=dict(color="white", size=15)),
    template="plotly_dark",
    height=650,
    hovermode="x unified"
)

fig.show()

corr = prices["EURGBP"].corr(prices["EURUSD"])
print(f"\n  Price correlation: {corr:.4f}")
print(f"  Mean spread:       {spread_mean:.4f}")
print(f"  Spread std dev:    {spread.std():.4f}")

In [ ]:
from statsmodels.tsa.stattools import coint, adfuller
import statsmodels.api as sm

# ============================================================
# ENGLE-GRANGER COINTEGRATION TEST
# ============================================================

score, pvalue, critical_values = coint(prices["EURGBP"], prices["EURUSD"])

print("=" * 50)
print("  Engle-Granger Cointegration Test")
print("=" * 50)
print(f"  Test statistic:  {score:.4f}")
print(f"  P-value:         {pvalue:.4f}")
print(f"  Critical values:")
print(f"    1%:  {critical_values[0]:.4f}")
print(f"    5%:  {critical_values[1]:.4f}")
print(f"    10%: {critical_values[2]:.4f}")
print("=" * 50)

if pvalue < 0.05:
    print(f"\n  ✅ COINTEGRATED at 95% confidence (p={pvalue:.4f})")
    print(f"  The spread is statistically likely to mean revert.")
    print(f"  Pairs trading is appropriate for EURGBP/EURUSD.")
else:
    print(f"\n  ❌ NOT cointegrated at 95% confidence (p={pvalue:.4f})")
    print(f"  Mean reversion is not statistically confirmed.")

# ============================================================
# HEDGE RATIO
# ============================================================

X = sm.add_constant(prices["EURUSD"])
model = sm.OLS(prices["EURGBP"], X).fit()
hedge_ratio = model.params["EURUSD"]
intercept   = model.params["const"]

print(f"\n  Hedge ratio (β):  {hedge_ratio:.4f}")
print(f"  Intercept:        {intercept:.4f}")
print(f"\n  Interpretation: For every 1 unit of EURGBP,")
print(f"  short {hedge_ratio:.4f} units of EURUSD to hedge the position.")

# ============================================================
# COINTEGRATED SPREAD
# ============================================================

spread_coint = prices["EURGBP"] - hedge_ratio * prices["EURUSD"] - intercept

# ============================================================
# ADF TEST
# ============================================================

adf_stat, adf_p, _, _, adf_critical, _ = adfuller(spread_coint)

print(f"\n{'=' * 50}")
print(f"  ADF Test on Cointegrated Spread")
print(f"{'=' * 50}")
print(f"  ADF statistic:   {adf_stat:.4f}")
print(f"  P-value:         {adf_p:.4f}")
print(f"  Critical values:")
for key, val in adf_critical.items():
    print(f"    {key}: {val:.4f}")

if adf_p < 0.05:
    print(f"\n  ✅ Spread is STATIONARY (p={adf_p:.4f})")
    print(f"  Mean reversion confirmed — safe to build signals.")
else:
    print(f"\n  ❌ Spread is NOT stationary (p={adf_p:.4f})")

# ============================================================
# PLOT COINTEGRATED SPREAD
# ============================================================

spread_mean = spread_coint.mean()
spread_std  = spread_coint.std()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=spread_coint.index, y=spread_coint,
    mode="lines", name="Cointegrated spread",
    line=dict(color="#28a745", width=1.2)
))

fig.add_hline(y=spread_mean, line_dash="dash",
              line_color="white", line_width=1.5,
              annotation_text="Mean",
              annotation_font_color="white")
fig.add_hline(y=spread_mean + spread_std, line_dash="dot",
              line_color="#fd7e14", line_width=1,
              annotation_text="+1σ", annotation_font_color="#fd7e14")
fig.add_hline(y=spread_mean - spread_std, line_dash="dot",
              line_color="#fd7e14", line_width=1,
              annotation_text="-1σ", annotation_font_color="#fd7e14")
fig.add_hline(y=spread_mean + 2*spread_std, line_dash="dot",
              line_color="#dc3545", line_width=1,
              annotation_text="+2σ", annotation_font_color="#dc3545")
fig.add_hline(y=spread_mean - 2*spread_std, line_dash="dot",
              line_color="#dc3545", line_width=1,
              annotation_text="-2σ", annotation_font_color="#dc3545")

fig.update_layout(
    title=dict(text="EURGBP/EURUSD Cointegrated Spread with σ Bands",
               x=0.5, font=dict(color="white", size=14)),
    xaxis_title="Date",
    yaxis_title="Spread",
    template="plotly_dark",
    height=450
)

fig.show()

In [ ]:
# ============================================================
# STEP 3 — Z-SCORE SIGNALS (FIXED)
# ============================================================

WINDOW = 30
ENTRY  = 2.0
EXIT   = 0.5

rolling_mean = spread_coint.rolling(WINDOW).mean()
rolling_std  = spread_coint.rolling(WINDOW).std()
zscore       = (spread_coint - rolling_mean) / rolling_std

signals = pd.DataFrame(index=prices.index)
signals["spread"]   = spread_coint
signals["zscore"]   = zscore
signals["position"] = 0

position = 0
positions = []

for i in range(len(signals)):
    z = signals["zscore"].iloc[i]

    if position == 0:
        if z > ENTRY:
            position = -1
        elif z < -ENTRY:
            position = 1
    elif position == 1:
        if z >= -EXIT:
            position = 0
    elif position == -1:
        if z <= EXIT:
            position = 0

    positions.append(position)

signals["position"] = positions

# --- Detect transitions using diff ---
pos_diff = pd.Series(positions, index=signals.index).diff()

long_entries  = signals[pos_diff == 1]
short_entries = signals[pos_diff == -1]
exits         = signals[(pos_diff != 0) & (pd.Series(positions, index=signals.index) == 0)]

# ============================================================
# PLOT
# ============================================================

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Cointegrated Spread with Trade Signals",
                    "Rolling Z-Score"),
    vertical_spacing=0.12,
    row_heights=[0.55, 0.45]
)

fig.add_trace(go.Scatter(
    x=signals.index, y=signals["spread"],
    mode="lines", name="Spread",
    line=dict(color="#28a745", width=1.2),
    hovertemplate="%{x}<br>Spread: %{y:.4f}<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=long_entries.index, y=long_entries["spread"],
    mode="markers", name="Long entry",
    marker=dict(color="#4a9edd", size=9, symbol="triangle-up"),
    hovertemplate="%{x}<br>Long entry<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=short_entries.index, y=short_entries["spread"],
    mode="markers", name="Short entry",
    marker=dict(color="#dc3545", size=9, symbol="triangle-down"),
    hovertemplate="%{x}<br>Short entry<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=exits.index, y=exits["spread"],
    mode="markers", name="Exit",
    marker=dict(color="yellow", size=7, symbol="x"),
    hovertemplate="%{x}<br>Exit<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=signals.index, y=signals["zscore"],
    mode="lines", name="Z-score",
    line=dict(color="#9b59b6", width=1.2),
    hovertemplate="%{x}<br>Z-score: %{y:.2f}<extra></extra>"
), row=2, col=1)

for level, color, label in [
    ( ENTRY, "#dc3545", f"+{ENTRY}σ entry"),
    (-ENTRY, "#dc3545", f"-{ENTRY}σ entry"),
    ( EXIT,  "white",   "Mean / Exit"),
]:
    fig.add_hline(y=level, line_dash="dot" if level != 0 else "dash",
                  line_color=color, line_width=1,
                  annotation_text=label,
                  annotation_font_color=color,
                  row=2, col=1)

fig.add_trace(go.Scatter(
    x=signals.index,
    y=pd.Series(positions, index=signals.index) * ENTRY,
    mode="lines", name="Position",
    line=dict(width=0),
    fill="tozeroy",
    fillcolor="rgba(100,149,237,0.1)",
    hoverinfo="skip"
), row=2, col=1)

fig.update_layout(
    title=dict(text="EURGBP/EURUSD Pairs Trading Signals",
               x=0.5, font=dict(color="white", size=15)),
    template="plotly_dark",
    height=700,
    hovermode="x unified"
)

fig.show()

n_long  = len(long_entries)
n_short = len(short_entries)
n_exits = len(exits)

print(f"\n  Signal Summary")
print(f"  {'='*35}")
print(f"  Long entries:    {n_long}")
print(f"  Short entries:   {n_short}")
print(f"  Exits:           {n_exits}")
print(f"  Total trades:    {n_long + n_short}")
print(f"  Currently:       {'LONG' if positions[-1] == 1 else 'SHORT' if positions[-1] == -1 else 'FLAT'}")

In [ ]:
# ============================================================
# STEP 3 — Z-SCORE SIGNALS (FIXED)
# ============================================================

WINDOW = 30
ENTRY  = 1.5
EXIT   = 0.0

rolling_mean = spread_coint.rolling(WINDOW).mean()
rolling_std  = spread_coint.rolling(WINDOW).std()
zscore       = (spread_coint - rolling_mean) / rolling_std

signals = pd.DataFrame(index=prices.index)
signals["spread"]   = spread_coint
signals["zscore"]   = zscore
signals["position"] = 0

position = 0
positions = []

for i in range(len(signals)):
    z = signals["zscore"].iloc[i]

    if position == 0:
        if z > ENTRY:
            position = -1
        elif z < -ENTRY:
            position = 1
    elif position == 1:
        if z >= EXIT:
            position = 0
    elif position == -1:
        if z <= EXIT:
            position = 0

    positions.append(position)

signals["position"] = positions

# --- Detect transitions using diff ---
pos_diff = pd.Series(positions, index=signals.index).diff()

long_entries  = signals[pos_diff == 1]
short_entries = signals[pos_diff == -1]
exits         = signals[(pos_diff != 0) & (pd.Series(positions, index=signals.index) == 0)]

# ============================================================
# PLOT
# ============================================================

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Cointegrated Spread with Trade Signals",
                    "Rolling Z-Score"),
    vertical_spacing=0.12,
    row_heights=[0.55, 0.45]
)

fig.add_trace(go.Scatter(
    x=signals.index, y=signals["spread"],
    mode="lines", name="Spread",
    line=dict(color="#28a745", width=1.2),
    hovertemplate="%{x}<br>Spread: %{y:.4f}<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=long_entries.index, y=long_entries["spread"],
    mode="markers", name="Long entry",
    marker=dict(color="#4a9edd", size=9, symbol="triangle-up"),
    hovertemplate="%{x}<br>Long entry<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=short_entries.index, y=short_entries["spread"],
    mode="markers", name="Short entry",
    marker=dict(color="#dc3545", size=9, symbol="triangle-down"),
    hovertemplate="%{x}<br>Short entry<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=exits.index, y=exits["spread"],
    mode="markers", name="Exit",
    marker=dict(color="yellow", size=7, symbol="x"),
    hovertemplate="%{x}<br>Exit<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=signals.index, y=signals["zscore"],
    mode="lines", name="Z-score",
    line=dict(color="#9b59b6", width=1.2),
    hovertemplate="%{x}<br>Z-score: %{y:.2f}<extra></extra>"
), row=2, col=1)

for level, color, label in [
    ( ENTRY, "#dc3545", f"+{ENTRY}σ entry"),
    (-ENTRY, "#dc3545", f"-{ENTRY}σ entry"),
    ( EXIT,  "white",   "Mean / Exit"),
]:
    fig.add_hline(y=level, line_dash="dot" if level != 0 else "dash",
                  line_color=color, line_width=1,
                  annotation_text=label,
                  annotation_font_color=color,
                  row=2, col=1)

fig.add_trace(go.Scatter(
    x=signals.index,
    y=pd.Series(positions, index=signals.index) * ENTRY,
    mode="lines", name="Position",
    line=dict(width=0),
    fill="tozeroy",
    fillcolor="rgba(100,149,237,0.1)",
    hoverinfo="skip"
), row=2, col=1)

fig.update_layout(
    title=dict(text="EURGBP/EURUSD Pairs Trading Signals",
               x=0.5, font=dict(color="white", size=15)),
    template="plotly_dark",
    height=700,
    hovermode="x unified"
)

fig.show()

n_long  = len(long_entries)
n_short = len(short_entries)
n_exits = len(exits)

print(f"\n  Signal Summary")
print(f"  {'='*35}")
print(f"  Long entries:    {n_long}")
print(f"  Short entries:   {n_short}")
print(f"  Exits:           {n_exits}")
print(f"  Total trades:    {n_long + n_short}")
print(f"  Currently:       {'LONG' if positions[-1] == 1 else 'SHORT' if positions[-1] == -1 else 'FLAT'}")

In [ ]:
# ============================================================
# STEP 4 — BACKTEST P&L
# ============================================================

# Make sure we're using 2σ/0.5σ parameters
WINDOW = 30
ENTRY  = 2.0
EXIT   = 0.5

# --- Daily spread changes ---
signals["spread_change"] = signals["spread"].diff()

# --- P&L: position * next day's spread change ---
signals["pnl_daily"] = signals["position"].shift(1) * signals["spread_change"]

# --- Cumulative P&L ---
signals["pnl_cumulative"] = signals["pnl_daily"].cumsum()

# --- Drawdown ---
rolling_max = signals["pnl_cumulative"].cummax()
signals["drawdown"] = signals["pnl_cumulative"] - rolling_max

# ============================================================
# PERFORMANCE METRICS
# ============================================================

total_pnl      = signals["pnl_daily"].sum()
sharpe         = (signals["pnl_daily"].mean() / signals["pnl_daily"].std()) * np.sqrt(252)
max_drawdown   = signals["drawdown"].min()
win_days       = (signals["pnl_daily"] > 0).sum()
loss_days      = (signals["pnl_daily"] < 0).sum()
win_rate       = win_days / (win_days + loss_days)
avg_win        = signals[signals["pnl_daily"] > 0]["pnl_daily"].mean()
avg_loss       = signals[signals["pnl_daily"] < 0]["pnl_daily"].mean()
profit_factor  = abs(avg_win / avg_loss)

print("=" * 50)
print("  EURGBP/EURUSD Pairs Trading — Backtest Results")
print("=" * 50)
print(f"  Total P&L:         {total_pnl:.4f}")
print(f"  Sharpe Ratio:      {sharpe:.4f}")
print(f"  Max Drawdown:      {max_drawdown:.4f}")
print(f"  Win Rate:          {win_rate:.1%}")
print(f"  Avg Win:           {avg_win:.4f}")
print(f"  Avg Loss:          {avg_loss:.4f}")
print(f"  Profit Factor:     {profit_factor:.4f}")
print(f"  Total Trades:      {n_long + n_short}")
print("=" * 50)

# ============================================================
# PLOT
# ============================================================

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        "Cumulative P&L",
        "Daily P&L",
        "Drawdown"
    ),
    vertical_spacing=0.1,
    row_heights=[0.5, 0.25, 0.25]
)

# --- Cumulative P&L ---
fig.add_trace(go.Scatter(
    x=signals.index,
    y=signals["pnl_cumulative"],
    mode="lines",
    name="Cumulative P&L",
    line=dict(color="#28a745", width=2),
    hovertemplate="%{x}<br>P&L: %{y:.4f}<extra></extra>"
), row=1, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="white",
              line_width=1, row=1, col=1)

# Shade profitable vs unprofitable regions
fig.add_trace(go.Scatter(
    x=signals.index,
    y=signals["pnl_cumulative"].clip(lower=0),
    mode="lines", line=dict(width=0),
    fill="tozeroy", fillcolor="rgba(40,167,69,0.15)",
    showlegend=False, hoverinfo="skip"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=signals.index,
    y=signals["pnl_cumulative"].clip(upper=0),
    mode="lines", line=dict(width=0),
    fill="tozeroy", fillcolor="rgba(220,53,69,0.15)",
    showlegend=False, hoverinfo="skip"
), row=1, col=1)

# --- Daily P&L bars ---
colors_daily = ["#28a745" if x >= 0 else "#dc3545"
                for x in signals["pnl_daily"].fillna(0)]

fig.add_trace(go.Bar(
    x=signals.index,
    y=signals["pnl_daily"],
    name="Daily P&L",
    marker_color=colors_daily,
    hovertemplate="%{x}<br>Daily P&L: %{y:.4f}<extra></extra>"
), row=2, col=1)

# --- Drawdown ---
fig.add_trace(go.Scatter(
    x=signals.index,
    y=signals["drawdown"],
    mode="lines",
    name="Drawdown",
    line=dict(color="#dc3545", width=1.2),
    fill="tozeroy",
    fillcolor="rgba(220,53,69,0.2)",
    hovertemplate="%{x}<br>Drawdown: %{y:.4f}<extra></extra>"
), row=3, col=1)

# --- Annotate max drawdown ---
max_dd_date = signals["drawdown"].idxmin()
fig.add_annotation(
    x=max_dd_date,
    y=max_drawdown,
    text=f"Max DD: {max_drawdown:.4f}",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#dc3545",
    font=dict(color="white", size=10),
    row=3, col=1
)

fig.update_layout(
    title=dict(
        text="EURGBP/EURUSD Pairs Trading — Backtest Results",
        x=0.5, font=dict(color="white", size=15)
    ),
    template="plotly_dark",
    height=750,
    hovermode="x unified",
    showlegend=True
)

fig.show()